# Task 3 — SmallCNN Child Experiments

This notebook starts from the completed five-fold SmallCNN baselines. It does **not** retrain them. Run All resolves their saved checkpoints as parents, then trains two separate one-factor children:

1. Gender: add random brightness from 0.85 to 1.15; keep ordinary cross-entropy.
2. Usage: keep augmentation off; change only to capped class-balanced cross-entropy.

Both children keep the same CNN, input, optimiser, schedule, 30 epochs, seed, folds, and final-checkpoint rule. Select a Colab GPU runtime before running.


## 1. Mount Drive and load the submitted branch

Drive supplies the teacher-data archive, completed baseline parents, registry, and persistent experiment output.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import zipfile

REPO_URL = "https://github.com/TrnLin/MLA2.git"
BRANCH = "task-3-gender-usage-classification"
REPO_DIR = Path("/content/MLA2")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT_DIR = DRIVE_MOUNT / "MyDrive/MLA2"
DATA_ZIP = DRIVE_PROJECT_DIR / "data/task3-data.zip"
DRIVE_TASK_DIR = DRIVE_PROJECT_DIR / "task3"
DRIVE_REGISTRY = DRIVE_TASK_DIR / "results/runs.csv"


def run_checked(command, *, cwd=None):
    command = [str(part) for part in command]
    print("$", " ".join(command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True)


In [ ]:
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("Connect this notebook to a Google Colab runtime first.") from exc

drive.mount(str(DRIVE_MOUNT), force_remount=False)

if (REPO_DIR / ".git").is_dir():
    remote_url = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True
    ).strip()
    if remote_url != REPO_URL:
        raise RuntimeError(f"{REPO_DIR} belongs to a different repository: {remote_url}")
    run_checked(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    run_checked(["git", "switch", BRANCH], cwd=REPO_DIR)
    dirty = subprocess.check_output(
        ["git", "status", "--porcelain"], cwd=REPO_DIR, text=True
    ).strip()
    if dirty:
        print("Local repository changes found; skipped the automatic update.")
    else:
        run_checked(["git", "merge", "--ff-only", f"origin/{BRANCH}"], cwd=REPO_DIR)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
else:
    run_checked(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR])

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print(f"Repository ready: {REPO_DIR}")
print(f"Branch: {BRANCH}")
print(f"Commit: {commit}")


## 2. Copy the teacher data onto the runtime disk

Training reads images from Colab's local disk. The archive keeps the repository folder structure.


In [ ]:
if not DATA_ZIP.is_file():
    raise FileNotFoundError(f"Dataset archive not found: {DATA_ZIP}")

teacher_dir = REPO_DIR / "data/raw/teacher"
required_files = (
    teacher_dir / "train/styles_train.csv",
    teacher_dir / "test/styles_prediction.csv",
)
image_suffixes = {".jpg", ".jpeg"}

with zipfile.ZipFile(DATA_ZIP) as archive:
    names = archive.namelist()
    unsafe_names = [name for name in names if Path(name).is_absolute() or ".." in Path(name).parts]
    if unsafe_names:
        raise RuntimeError("The dataset archive contains an unsafe path.")
    expected_images = sum(
        name.startswith("data/raw/teacher/") and Path(name).suffix.lower() in image_suffixes
        for name in names
    )
    if expected_images == 0:
        raise RuntimeError("The archive has no teacher images in the expected folder.")
    current_images = sum(path.suffix.lower() in image_suffixes for path in teacher_dir.rglob("*"))
    needs_extract = current_images != expected_images or not all(path.is_file() for path in required_files)
    if needs_extract:
        print(f"Extracting {expected_images:,} teacher images...", flush=True)
        archive.extractall(REPO_DIR)
    else:
        print("Teacher data is already extracted; skipping.")

actual_images = sum(path.suffix.lower() in image_suffixes for path in teacher_dir.rglob("*"))
missing_files = [str(path) for path in required_files if not path.is_file()]
if actual_images != expected_images or missing_files:
    raise RuntimeError(
        f"Dataset check failed: expected {expected_images:,} images, found {actual_images:,}; "
        f"missing files: {missing_files}"
    )
print(f"Teacher data ready: {actual_images:,} images")


## 3. Resolve the saved parents and verify both children

This finds the newest complete baseline checkpoint for each fold. The checks verify the GPU, data, exact CNN, parent chain, and one-factor locks without taking an optimiser step.


In [ ]:
os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)

for output_dir in (DRIVE_TASK_DIR / "experiments", DRIVE_TASK_DIR / "logs", DRIVE_TASK_DIR / "results"):
    output_dir.mkdir(parents=True, exist_ok=True)

from fashion.train.task3_experiments import (
    check_task3_child_setup,
    latest_completed_baseline_parent_run_ids,
)

gender_parent_run_ids = latest_completed_baseline_parent_run_ids(
    "gender", output_root=DRIVE_TASK_DIR
)
usage_parent_run_ids = latest_completed_baseline_parent_run_ids(
    "usage", output_root=DRIVE_TASK_DIR
)
gender_child_check = check_task3_child_setup(
    "gender_brightness", parent_run_ids=gender_parent_run_ids, root=REPO_DIR, device_name="cuda"
)
usage_child_check = check_task3_child_setup(
    "usage_class_balanced", parent_run_ids=usage_parent_run_ids, root=REPO_DIR, device_name="cuda"
)

print("GPU:", gender_child_check["environment"]["gpu"])
print("Gender parents:", gender_parent_run_ids)
print("Usage parents: ", usage_parent_run_ids)
print("Gender change:", gender_child_check["changed_factor"])
print("Usage change: ", usage_child_check["changed_factor"])
print("Optimizer steps during checks:", gender_child_check["optimizer_steps"], usage_child_check["optimizer_steps"])


## 4. Train the Gender brightness child

This foreground cell trains folds 0–4. Only training brightness changes from its matching Gender baseline parent.


In [ ]:
from fashion.train.task3_experiments import run_task3_child_cv

gender_child_result = run_task3_child_cv(
    "gender_brightness",
    parent_run_ids=gender_parent_run_ids,
    folds=range(5),
    root=REPO_DIR,
    output_root=DRIVE_TASK_DIR,
    registry_path=DRIVE_REGISTRY,
    registry_mirrors=[REPO_DIR / "results/runs.csv"],
    device_name="cuda",
)
gender_child_result


## 5. Train the Usage class-balanced-loss child

This foreground cell trains folds 0–4. Only the loss weights change from its matching Usage baseline parent.


In [ ]:
usage_child_result = run_task3_child_cv(
    "usage_class_balanced",
    parent_run_ids=usage_parent_run_ids,
    folds=range(5),
    root=REPO_DIR,
    output_root=DRIVE_TASK_DIR,
    registry_path=DRIVE_REGISTRY,
    registry_mirrors=[REPO_DIR / "results/runs.csv"],
    device_name="cuda",
)
usage_child_result


## 6. Compare each child with its own baseline

This is only a first result table. The main Task 3 notebook must still inspect curves, classes, mistakes, robustness, and the prewritten pass/fail rules before accepting a child.


In [ ]:
import pandas as pd

baseline_metrics = {
    target: json.loads(
        (DRIVE_TASK_DIR / "baseline" / target / "aggregate/metrics.json").read_text(encoding="utf-8")
    )
    for target in ("gender", "usage")
}
comparison = []
for target, child in (("gender", gender_child_result), ("usage", usage_child_result)):
    parent = baseline_metrics[target]
    comparison.append({
        "target": target,
        "baseline_macro_f1": parent["macro_f1"],
        "child_macro_f1": child["metrics"]["macro_f1"],
        "macro_f1_change": child["metrics"]["macro_f1"] - parent["macro_f1"],
        "child_metrics_path": child["metrics_path"],
    })
pd.DataFrame(comparison)
